In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Imports

In [111]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from roar import DATA_DIR as DATA_ROOT
from roar import MIC_CHANNELS
from roar.preprocessing import (
    extract_features_from_h5_file,
    fix_channel_names,
    get_channel_mapping,
    parse_filename,
)

### Mappings

In [33]:
TRACKS = ["track150", "track211"]
CARS = ["01_ID.4", "02_Q8 e-tron", "03_Taycan", "04_E-Golf"]
TYRES = ["tyre1", "tyre3", "tyre6", "tyre10", "tyre12", "tyre13"]

In [34]:
# Set the data root
if not DATA_ROOT.exists():
    DATA_ROOT = Path(r"C:\Users\Lars\Büro\KIT\Master\WS_25_26\AIFB_Seminar\projects\workspace\data")

SYNONYM_TO_CHANNEL = get_channel_mapping()

h5_paths = list(DATA_ROOT.rglob("*.h5"))  # pyright: ignore[reportPossiblyUnboundVariable]
print(f"Found {len(h5_paths)} files")

for h5_path in h5_paths:
    fix_channel_names(file_path=h5_path, mapping=SYNONYM_TO_CHANNEL, verbose=False)

Found 204 files


### Build dataset

In [35]:
# Get the h5 file paths

h5_paths = list(DATA_ROOT.rglob("*.h5"))
print(f"Found {len(h5_paths)} files")

rows = []

for path in h5_paths:
    try:
        feat_dict = extract_features_from_h5_file(path, channels=MIC_CHANNELS)
        if not feat_dict:
            print(f"No usable mic data in {path}, skipping.")
            continue  # no usable mic data

        meta = parse_filename(path)

        row = {
            **feat_dict,
            **meta,
        }
        rows.append(row)

    except Exception as e:
        print(f"Error processing {path}: {e}")

df = pd.DataFrame(rows)

df.info(verbose=True, show_counts=True)

Found 204 files


/Users/moritzfeik/Developer/ROAR/src/roar/preprocessing/load_data.py:93: UserWarning: Channel 'NAWSSound' not found in /Users/moritzfeik/Developer/ROAR/data_cleaned/track211/02_Q8 e-tron/tyre12/track211_Q8 e-tron_tyre12_meas5_2p5_1_2025-08-15_12-13-33.h5.
  warn(f"Channel '{channel_name}' not found in {file_path}.")
/Users/moritzfeik/Developer/ROAR/src/roar/preprocessing/load_data.py:93: UserWarning: Channel 'NAWSSound' not found in /Users/moritzfeik/Developer/ROAR/data_cleaned/track211/02_Q8 e-tron/tyre12/track211_Q8 e-tron_tyre12_meas6_2p5_1_2025-08-15_12-34-39.h5.
  warn(f"Channel '{channel_name}' not found in {file_path}.")
/Users/moritzfeik/Developer/ROAR/src/roar/preprocessing/load_data.py:93: UserWarning: Channel 'NAWSSound' not found in /Users/moritzfeik/Developer/ROAR/data_cleaned/track211/02_Q8 e-tron/tyre12/track211_Q8 e-tron_tyre12_meas6_2p5_1_2025-08-15_12-30-46.h5.
  warn(f"Channel '{channel_name}' not found in {file_path}.")
/Users/moritzfeik/Developer/ROAR/src/roar/prep

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 203 columns):
 #    Column                      Non-Null Count  Dtype         
---   ------                      --------------  -----         
 0    Ch_1_labV12_rms             204 non-null    float64       
 1    Ch_1_labV12_mean            204 non-null    float64       
 2    Ch_1_labV12_std             204 non-null    float64       
 3    Ch_1_labV12_max             204 non-null    float64       
 4    Ch_1_labV12_crest           204 non-null    float64       
 5    Ch_1_labV12_zcr             204 non-null    float64       
 6    Ch_1_labV12_spec_centroid   204 non-null    float64       
 7    Ch_1_labV12_spec_rolloff    204 non-null    float64       
 8    Ch_1_labV12_spec_flatness   204 non-null    float64       
 9    Ch_1_labV12_spec_bandwidth  204 non-null    float64       
 10   Ch_1_labV12_band_0          204 non-null    float64       
 11   Ch_1_labV12_band_1          204 non-null   

### Fix missing values

1. mic_iso channels 151/204 non-null

    track211 ID.4 tyre3 is missing mic iso, remove it completely for now, contains however "MikrofonOst" & "MikrofonWest" with similar sample rates (btw regular files also contain a "mic_2m" additionally to "mic_iso") which could be used.  
    These channels are actually what we need -> this problem is fixed, in fix_channel_names this is taken care of

2. NAWSSound channels 199/204 non-null

    5 track211 Q8 e-tron tyre12 files are missing the NAWS channel, fill it with means of the respective group

In [43]:
df_clean = df.copy()

# Fill missing NAWSSound features with group-wise mean imputation
missing_cols = [col for col in df_clean.columns if col.startswith("NAWSSound")]
group_cols = ["track_ID", "vehicle", "tyre_ID"]

for col in missing_cols:
    df_clean[col] = df_clean.groupby(group_cols)[col].transform(lambda g: g.fillna(g.mean()))

# If some groups have only NaN (rare), fill remaining with global mean
if df_clean[missing_cols].isna().any().any():
    print("Some NAWSSound groups have only NaN, filling with global mean.")
    df_clean[missing_cols] = df_clean[missing_cols].fillna(df_clean[missing_cols].mean())

df_clean.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 203 columns):
 #    Column                      Non-Null Count  Dtype         
---   ------                      --------------  -----         
 0    Ch_1_labV12_rms             204 non-null    float64       
 1    Ch_1_labV12_mean            204 non-null    float64       
 2    Ch_1_labV12_std             204 non-null    float64       
 3    Ch_1_labV12_max             204 non-null    float64       
 4    Ch_1_labV12_crest           204 non-null    float64       
 5    Ch_1_labV12_zcr             204 non-null    float64       
 6    Ch_1_labV12_spec_centroid   204 non-null    float64       
 7    Ch_1_labV12_spec_rolloff    204 non-null    float64       
 8    Ch_1_labV12_spec_flatness   204 non-null    float64       
 9    Ch_1_labV12_spec_bandwidth  204 non-null    float64       
 10   Ch_1_labV12_band_0          204 non-null    float64       
 11   Ch_1_labV12_band_1          204 non-null   

### Transform to Categorical

In [49]:
from roar import MEASUREMENTS_CLEAN_NAMES

df_clean["measure"] = df_clean["measure"].replace(MEASUREMENTS_CLEAN_NAMES).astype("category")
df_clean["track_ID"] = df_clean["track_ID"].astype("category")
df_clean["vehicle"] = df_clean["vehicle"].astype("category")

### Create LOGO Split

In [50]:
# Explicitly exclude non-feature columns (safer than dtype selection alone)
non_feature_cols = ["track_ID", "vehicle", "tyre_ID", "file_stem", "file_path", "measure", "date"]
feature_cols = [
    c for c in df_clean.columns if c not in non_feature_cols and df_clean[c].dtype == "float64"
]

print("Number of features:", len(feature_cols))

X = df_clean[feature_cols].to_numpy()
y = df_clean["track_ID"].to_numpy()
groups = df_clean["vehicle"].to_numpy()

# Encode string labels to integers for XGBoost
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Classes:", le.classes_)

# LOGO splitter
logo = LeaveOneGroupOut()

Number of features: 196
Classes: [150 211]


### Wrapper to adapt the threshold
Works if model has predict_proba so we can tune that on the validation set

In [59]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone


class ThresholdClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold
        self.response_method = "predict_proba"

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)

    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

### Logistic Regression with PCA

In [105]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),  # standardize features
        ("pca", PCA(n_components=0.95)),  # reduce to top 5 PCs
        (
            "clf",
            ThresholdClassifier(
                estimator=LogisticRegression(random_state=42, max_iter=1000), threshold=0.5
            ),
        ),
    ]
)

param_grid = {
    "pca__n_components": [0.8, 0.9, 0.95, 0.99, 0.999],
    "clf__threshold": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    "clf__estimator__C": [0.1, 1, 10],
    "clf__estimator__class_weight": [None, "balanced"],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)
pd.DataFrame(grid.cv_results_)[
    [
        "param_pca__n_components",
        "param_clf__threshold",
        "param_clf__estimator__C",
        "param_clf__estimator__class_weight",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 270 candidates, totalling 1080 fits


,param_pca__n_components,param_clf__threshold,param_clf__estimator__C,param_clf__estimator__class_weight,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
129,0.999,0.8,1.0,None,0.796309,0.090683,0.794681,0.081476
169,0.999,0.7,1.0,balanced,0.790687,0.072446,0.784681,0.066573
128,0.990,0.8,1.0,None,0.789184,0.067322,0.786004,0.064335
168,0.990,0.7,1.0,balanced,0.789184,0.067322,0.786004,0.064335
164,0.999,0.6,1.0,balanced,0.789045,0.075832,0.788649,0.064594


### Random Forest

In [108]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
)
threshold_rf = ThresholdClassifier(estimator=rf, threshold=0.5)

param_grid = {
    "threshold": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    "estimator__max_depth": [None, 5, 10],
    "estimator__min_samples_leaf": [1, 5, 10],
    "estimator__class_weight": [None, "balanced", "balanced_subsample"],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=threshold_rf,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)
pd.DataFrame(grid.cv_results_)[
    [
        "param_estimator__max_depth",
        "param_estimator__min_samples_leaf",
        "param_estimator__class_weight",
        "param_threshold",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 243 candidates, totalling 972 fits


,param_estimator__max_depth,param_estimator__min_samples_leaf,param_estimator__class_weight,param_threshold,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
32,5,1,None,0.6,0.761743,0.141837,0.798639,0.089427
5,None,1,None,0.6,0.748120,0.145239,0.777316,0.100974
59,10,1,None,0.6,0.748120,0.145239,0.777316,0.100974
168,None,1,balanced_subsample,0.7,0.745078,0.107107,0.756413,0.081336
222,10,1,balanced_subsample,0.7,0.745078,0.107107,0.756413,0.081336


### XGBoost

In [102]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    n_jobs=-1,
    eval_metric="logloss",
    random_state=42,
)

threshold_xgb = ThresholdClassifier(estimator=xgb, threshold=0.5)

param_grid = {
    "threshold": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    "estimator__max_depth": [3, 5, 7],
    "estimator__min_child_weight": [1, 5],
    "estimator__learning_rate": [0.01, 0.05, 0.1],
    "estimator__gamma": [0, 0.1],
    "estimator__subsample": [0.8, 1.0],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=threshold_xgb,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)
pd.DataFrame(grid.cv_results_)[
    [
        "param_threshold",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 648 candidates, totalling 2592 fits


,param_threshold,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
258,0.7,0.772855,0.135990,0.787475,0.113799
294,0.7,0.772855,0.135990,0.787475,0.113799
223,0.8,0.764118,0.144537,0.768553,0.121788
582,0.7,0.757759,0.151009,0.769120,0.136130
618,0.7,0.757759,0.151009,0.769120,0.136130


### LightGBM

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="X does not have valid feature names*")

In [115]:
lgbm = LGBMClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

threshold_lgbm = ThresholdClassifier(estimator=lgbm, threshold=0.5)

param_grid = {
    "threshold": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    "estimator__num_leaves": [31, 50, 100],
    "estimator__max_depth": [5, 10, 20],
    "estimator__learning_rate": [0.01, 0.05, 0.1],
    "estimator__class_weight": [None, "balanced"],
}

scoring = {
    "weighted_f1": "f1_weighted",
    "acc": "accuracy",
}

grid = GridSearchCV(
    estimator=threshold_lgbm,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=1,
)
grid.fit(X, y_enc)
pd.DataFrame(grid.cv_results_)[
    [
        "param_threshold",
        "param_estimator__num_leaves",
        "param_estimator__max_depth",
        "param_estimator__learning_rate",
        "param_estimator__learning_rate",
        "param_estimator__class_weight",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 486 candidates, totalling 1944 fits


python(1833) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(1834) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(1835) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(1836) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(1837) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(1838) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(1839) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(1840) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


,param_threshold,param_estimator__num_leaves,param_estimator__max_depth,param_estimator__learning_rate,param_estimator__learning_rate,param_estimator__class_weight,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
356,0.6,31,10,0.05,0.05,balanced,0.753808,0.142869,0.775488,0.121232
365,0.6,50,10,0.05,0.05,balanced,0.753808,0.142869,0.775488,0.121232
374,0.6,100,10,0.05,0.05,balanced,0.753808,0.142869,0.775488,0.121232
383,0.6,31,20,0.05,0.05,balanced,0.753808,0.142869,0.775488,0.121232
392,0.6,50,20,0.05,0.05,balanced,0.753808,0.142869,0.775488,0.121232


### TabFPM

In [109]:
from tabpfn import TabPFNClassifier

# For this to work you must accept the license terms at https://huggingface.co/Prior-Labs/tabpfn_2_5
# and be logged in via the huggingface_hub CLI.

tabpfn = TabPFNClassifier()

param_grid = {"balance_probabilities": [True, False], "n_estimators": [2, 4, 8, 16]}

grid = GridSearchCV(
    estimator=tabpfn,
    param_grid=param_grid,
    scoring=scoring,
    refit="weighted_f1",
    cv=logo.split(X, y_enc, groups),
    n_jobs=-1,
    verbose=2,
)
grid.fit(X, y_enc)
pd.DataFrame(grid.cv_results_)[
    [
        "param_balance_probabilities",
        "param_n_estimators",
        "mean_test_weighted_f1",
        "std_test_weighted_f1",
        "mean_test_acc",
        "std_test_acc",
    ]
].nlargest(5, columns="mean_test_weighted_f1")

Fitting 4 folds for each of 8 candidates, totalling 32 fits
[CV] END .........balance_probabilities=True, n_estimators=2; total time=  24.0s
[CV] END .........balance_probabilities=True, n_estimators=2; total time=  47.2s
[CV] END .........balance_probabilities=True, n_estimators=2; total time=  47.7s
[CV] END .........balance_probabilities=True, n_estimators=2; total time=  48.3s
[CV] END .........balance_probabilities=True, n_estimators=4; total time=  54.0s
[CV] END .........balance_probabilities=True, n_estimators=4; total time=  54.6s
[CV] END .........balance_probabilities=True, n_estimators=4; total time=  54.8s
[CV] END .........balance_probabilities=True, n_estimators=4; total time=  56.4s
[CV] END .........balance_probabilities=True, n_estimators=8; total time= 1.0min
[CV] END .........balance_probabilities=True, n_estimators=8; total time=  45.9s
[CV] END .........balance_probabilities=True, n_estimators=8; total time=  50.2s
[CV] END .........balance_probabilities=True, n_e

,param_balance_probabilities,param_n_estimators,mean_test_weighted_f1,std_test_weighted_f1,mean_test_acc,std_test_acc
2,True,8,0.845811,0.083166,0.842525,0.086928
3,True,16,0.822421,0.090209,0.825753,0.084070
1,True,4,0.819800,0.100593,0.823107,0.096375
4,False,2,0.800943,0.171826,0.828985,0.131151
7,False,16,0.800943,0.171826,0.828985,0.131151


### Balancing Helper

In [51]:
def car_balanced_weights(vehicles):
    counts = Counter(vehicles)
    w = np.array([1.0 / counts[c] for c in vehicles], dtype=float)
    # normalize for nicer scales (optional)
    return w * (len(w) / w.sum())

### Training Loop

In [52]:
# Config
CLASS_BALANCING = False
CAR_BALANCING = True
THRESHOLD_TUNING = True  # Only for Random Forest

In [ ]:
rf_accs, rf_f1s = [], []
xgb_accs, xgb_f1s = [], []
lgbm_accs, lgbm_f1s = [], []
tabpfn_accs, tabpfn_f1s = [], []
fold_results = []

# Optional: to accumulate global confusion matrices across folds
# (nice for seminar slides)
labels = le.classes_
rf_cm_total = np.zeros((len(labels), len(labels)), dtype=int)
xgb_cm_total = np.zeros_like(rf_cm_total)
lgbm_cm_total = np.zeros_like(rf_cm_total)
tabpfn_cm_total = np.zeros_like(rf_cm_total)

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups), start=1):
    held_out_car = groups[test_idx][0]
    print(f"\n===== LOGO Fold {fold} | Held-out car: {held_out_car} =====")
    print(f"Train n={len(train_idx)}, Test n={len(test_idx)}")

    # Split data (enc for XGBoost)
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    y_train_enc, y_test_enc = y_enc[train_idx], y_enc[test_idx]

    # Compute class weights
    class_weights = compute_sample_weight("balanced", y_train)

    # Compute car-based weights to balance training samples by car
    car_train = df_clean.iloc[train_idx]["vehicle"].to_numpy()
    car_weights = car_balanced_weights(car_train)

    # ------------------ Random Forest ------------------
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=42,
    )

    # Fit with appropriate sample weights
    if CLASS_BALANCING and CAR_BALANCING:
        combined_weights = class_weights * car_weights
        combined_weights = combined_weights * (len(combined_weights) / combined_weights.sum())
        rf.fit(X_train, y_train, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        rf.fit(X_train, y_train, sample_weight=class_weights)
    elif CAR_BALANCING:
        rf.fit(X_train, y_train, sample_weight=car_weights)
    else:
        rf.fit(X_train, y_train)

    # Threshold tuning
    if THRESHOLD_TUNING:
        proba0 = rf.predict_proba(X_test)[:, 0]
        t0 = 0.3  # threshold for class 0
        y_pred_rf_enc = (proba0 < t0).astype(int)
        y_pred_rf = le.inverse_transform(y_pred_rf_enc)
    else:
        y_pred_rf = rf.predict(X_test)

    rf_acc = accuracy_score(y_test, y_pred_rf)
    rf_f1 = f1_score(y_test, y_pred_rf, average="weighted")

    rf_accs.append(rf_acc)
    rf_f1s.append(rf_f1)

    cm_rf = confusion_matrix(y_test, y_pred_rf, labels=labels)
    rf_cm_total += cm_rf

    print(f"RF  - acc: {rf_acc:.3f}, f1: {rf_f1:.3f}")
    print(cm_rf)

    # ------------------ XGBoost ------------------
    # For 2-class problems we can use binary:logistic.
    # If we ever add more road types, switch to multi:softprob + num_class.
    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        n_jobs=-1,
        eval_metric="logloss",
        random_state=42,
    )

    if CLASS_BALANCING and CAR_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=class_weights)
    elif CAR_BALANCING:
        xgb.fit(X_train, y_train_enc, sample_weight=car_weights)
    else:
        xgb.fit(X_train, y_train_enc)

    y_pred_xgb_enc = xgb.predict(X_test)
    y_pred_xgb = le.inverse_transform(y_pred_xgb_enc.astype(int))

    xgb_acc = accuracy_score(y_test, y_pred_xgb)
    xgb_f1 = f1_score(y_test, y_pred_xgb, average="weighted")

    xgb_accs.append(xgb_acc)
    xgb_f1s.append(xgb_f1)

    cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=labels)
    xgb_cm_total += cm_xgb

    print(f"XGB - acc: {xgb_acc:.3f}, f1: {xgb_f1:.3f}")
    print(cm_xgb)

    # ------------------ LightGBM ------------------
    lgbm = LGBMClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )

    if CLASS_BALANCING and CAR_BALANCING:
        lgbm.fit(X_train, y_train_enc, sample_weight=combined_weights)
    elif CLASS_BALANCING:
        lgbm.fit(X_train, y_train_enc, sample_weight=class_weights)
    elif CAR_BALANCING:
        lgbm.fit(X_train, y_train_enc, sample_weight=car_weights)
    else:
        lgbm.fit(X_train, y_train_enc)

    if THRESHOLD_TUNING:
        proba0 = lgbm.predict_proba(X_test)[:, 0]
        t0 = 0.3  # threshold for class 0
        y_pred_lgbm_enc = (proba0 < t0).astype(int)
        y_pred_lgbm = le.inverse_transform(y_pred_lgbm_enc)
    else:
        y_pred_lgbm_enc = lgbm.predict(X_test)
        y_pred_lgbm = le.inverse_transform(y_pred_lgbm_enc.astype(int))

    lgbm_acc = accuracy_score(y_test, y_pred_lgbm)
    lgbm_f1 = f1_score(y_test, y_pred_lgbm, average="weighted")

    lgbm_accs.append(lgbm_acc)
    lgbm_f1s.append(lgbm_f1)

    cm_lgbm = confusion_matrix(y_test, y_pred_lgbm, labels=labels)
    lgbm_cm_total += cm_lgbm

    print(f"LGBM - acc: {lgbm_acc:.3f}, f1: {lgbm_f1:.3f}")
    print(cm_lgbm)

    # ------------------ TabPFN ------------------
    tabpfn = TabPFNClassifier()
    tabpfn.fit(X_train, y_train_enc)

    if THRESHOLD_TUNING:
        proba0 = tabpfn.predict_proba(X_test)[:, 0]
        t0 = 0.3  # threshold for class 0
        y_pred_tabpfn_enc = (proba0 < t0).astype(int)
        y_pred_tabpfn = le.inverse_transform(y_pred_tabpfn_enc)
    else:
        y_pred_tabpfn_enc = tabpfn.predict(X_test)
    y_pred_tabpfn = le.inverse_transform(y_pred_tabpfn_enc.astype(int))

    tabpfn_acc = accuracy_score(y_test, y_pred_tabpfn)
    tabpfn_f1 = f1_score(y_test, y_pred_tabpfn, average="weighted")

    tabpfn_accs.append(tabpfn_acc)
    tabpfn_f1s.append(tabpfn_f1)

    cm_tabpfn = confusion_matrix(y_test, y_pred_tabpfn, labels=labels)
    tabpfn_cm_total += cm_tabpfn

    print(f"TabPFN - acc: {tabpfn_acc:.3f}, f1: {tabpfn_f1:.3f}")
    print(cm_tabpfn)

    fold_results.append(
        {
            "fold": fold,
            "held_out_car": held_out_car,
            "n_test": len(test_idx),
            "rf_acc": rf_acc,
            "rf_f1": rf_f1,
            "xgb_acc": xgb_acc,
            "xgb_f1": xgb_f1,
            "lgbm_acc": lgbm_acc,
            "lgbm_f1": lgbm_f1,
            "tabpfn_acc": tabpfn_acc,
            "tabpfn_f1": tabpfn_f1,
        }
    )


# --------------- Summary ---------------
print("\n===== LOGO Summary (per held-out car) =====")
for r in fold_results:
    print(
        f"{r['held_out_car']}: RF acc={r['rf_acc']:.3f}, XGB acc={r['xgb_acc']:.3f}, "
        f"LGBM acc={r['lgbm_acc']:.3f}, TabPFN acc={r['tabpfn_acc']:.3f}"
    )

print("\n===== Overall (mean ± std) =====")
print(
    f"RF  acc: {np.mean(rf_accs):.3f} ± {np.std(rf_accs):.3f} | "
    f"f1: {np.mean(rf_f1s):.3f} ± {np.std(rf_f1s):.3f}"
)
print(
    f"XGB acc: {np.mean(xgb_accs):.3f} ± {np.std(xgb_accs):.3f} | "
    f"f1: {np.mean(xgb_f1s):.3f} ± {np.std(xgb_f1s):.3f}"
)
print(
    f"LGBM acc: {np.mean(lgbm_accs):.3f} ± {np.std(lgbm_accs):.3f} | "
    f"f1: {np.mean(lgbm_f1s):.3f} ± {np.std(lgbm_f1s):.3f}"
)
print(
    f"TabPFN acc: {np.mean(tabpfn_accs):.3f} ± {np.std(tabpfn_accs):.3f} | "
    f"f1: {np.mean(tabpfn_f1s):.3f} ± {np.std(tabpfn_f1s):.3f}"
)

print("\nRF total confusion matrix (rows=true, cols=pred):")
print(rf_cm_total)
print("\nXGB total confusion matrix (rows=true, cols=pred):")
print(xgb_cm_total)
print("\nLGBM total confusion matrix (rows=true, cols=pred):")
print(lgbm_cm_total)
print("\nTabPFN total confusion matrix (rows=true, cols=pred):")
print(tabpfn_cm_total)
print("\nLabel order:", labels)


===== LOGO Fold 1 | Held-out car: E-Golf =====
Train n=179, Test n=25
RF  - acc: 0.480, f1: 0.384
[[10  0]
 [13  2]]
XGB - acc: 0.640, f1: 0.534
[[ 1  9]
 [ 0 15]]
TabPFN - acc: 0.760, f1: 0.729
[[ 4  6]
 [ 0 15]]

===== LOGO Fold 2 | Held-out car: ID.4 =====
Train n=115, Test n=89
RF  - acc: 0.775, f1: 0.811
[[ 5  4]
 [16 64]]
XGB - acc: 0.944, f1: 0.939
[[ 5  4]
 [ 1 79]]
TabPFN - acc: 0.966, f1: 0.968
[[ 9  0]
 [ 3 77]]

===== LOGO Fold 3 | Held-out car: Q8 e-tron =====
Train n=141, Test n=63
RF  - acc: 0.714, f1: 0.728
[[16  2]
 [16 29]]
XGB - acc: 0.730, f1: 0.736
[[11  7]
 [10 35]]
TabPFN - acc: 0.762, f1: 0.773
[[18  0]
 [15 30]]

===== LOGO Fold 4 | Held-out car: Taycan =====
Train n=177, Test n=27
RF  - acc: 0.593, f1: 0.597
[[ 4  5]
 [ 6 12]]
XGB - acc: 0.593, f1: 0.496
[[ 0  9]
 [ 2 16]]
TabPFN - acc: 0.741, f1: 0.737
[[ 5  4]
 [ 3 15]]

===== LOGO Summary (per held-out car) =====
E-Golf: RF acc=0.480, XGB acc=0.640, TabPFN acc=0.760
ID.4: RF acc=0.775, XGB acc=0.944, TabPF

### Feature Importance

In [ ]:
rf_final = RandomForestClassifier(n_estimators=500, max_depth=None, n_jobs=-1, random_state=42)

# Compute class weights
class_weights_final = compute_sample_weight("balanced", y)

# Compute car-based weights to balance training samples by car
car_final = df_clean["vehicle"].to_numpy()
car_weights_final = car_balanced_weights(car_final)
if CLASS_BALANCING and CAR_BALANCING:
    combined_weights_final = class_weights_final * car_weights_final
    combined_weights_final = combined_weights_final * (
        len(combined_weights_final) / combined_weights_final.sum()
    )
    rf_final.fit(X, y, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    rf_final.fit(X, y, sample_weight=class_weights_final)
elif CAR_BALANCING:
    rf_final.fit(X, y, sample_weight=car_weights_final)
else:
    rf_final.fit(X, y)

importances_rf = pd.Series(rf_final.feature_importances_, index=feature_cols)
print("\nTop 20 features Random Forest:")
print(importances_rf.sort_values(ascending=False).head(20))

In [ ]:
xgb_final = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    n_jobs=-1,
    eval_metric="logloss",
    random_state=42,
)

if CLASS_BALANCING and CAR_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=class_weights_final)
elif CAR_BALANCING:
    xgb_final.fit(X, y_enc, sample_weight=car_weights_final)
else:
    xgb_final.fit(X, y_enc)

xgb_feature_scores = xgb_final.get_booster().get_score(importance_type="gain")

# Convert to pandas Series
importances_xgb = (
    pd.Series(xgb_feature_scores)
    .rename(index=lambda x: feature_cols[int(x[1:])])  # maps f0 → feature name
    .sort_values(ascending=False)
)

xgb_norm = importances_xgb / importances_xgb.sum()
print("\nTop 20 features XGBoost:")
print(xgb_norm.head(20))

In [ ]:
lgbm_final = LGBMClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

if CLASS_BALANCING and CAR_BALANCING:
    lgbm_final.fit(X, y_enc, sample_weight=combined_weights_final)
elif CLASS_BALANCING:
    lgbm_final.fit(X, y_enc, sample_weight=class_weights_final)
elif CAR_BALANCING:
    lgbm_final.fit(X, y_enc, sample_weight=car_weights_final)
else:
    lgbm_final.fit(X, y_enc)

importances_lgbm = pd.Series(lgbm_final.feature_importances_, index=feature_cols)
print("\nTop 20 features LightGBM:")
print(importances_lgbm.sort_values(ascending=False).head(20))